# Week 4 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-04.md`](../../weeks/week-04.md) · **Given engine:** [`rsa_lab.py`](rsa_lab.py)

Teaching walkthrough of the week-4 studio. It **imports the reference attacks from
[`solution.py`](solution.py)** — never re-pasting them — so what runs here is exactly
what `test_rsa.py` grades. Correctness is verified separately:
`python3 studios/_verify_solutions.py week-04`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

Every break this week leaves RSA's math intact. We attack the *conditions* RSA
depends on — good independent entropy, and constant-time secret handling — never a
strong modulus.

In [ ]:
# --- bootstrap: week folder (for solution/rsa_lab) + repo root (for seclab) ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-04"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import rsa_lab as lab
import solution
corpus = lab.load_keys()   # 8 public keys; any one looks fine

## The reduction, in miniature

To get the private exponent `d` you need `φ`; to get `φ` you need the factorization
of `n`. For a 2048-bit `n` that is infeasible — **provided** `p` and `q` were good.
Tasks 2–3 attack that proviso, not the arithmetic.

In [ ]:
pub, priv = lab.rsa_keygen(61, 53, e=17)
c = lab.encrypt(42, pub)
assert lab.decrypt(c, priv) == 42
n, e = pub
d = lab.factor_from_shared(n, 61, e=17)   # knowing a factor yields d
assert lab.decrypt(c, (n, d)) == 42
print("RSA round-trips; d is recoverable once n is factored (the whole reduction)")

## Task 2 — shared-factor recovery (batch-GCD): the *Ps and Qs* attack

Each modulus is fine in isolation. But a weak RNG made two keys share a prime, and
`gcd(n_i, n_j)` reveals it *instantly* — no factoring. `batch_gcd_recover` scans
every pair; a `gcd != 1` is the shared prime, and `factor_from_shared` turns it
into `d` for **both** keys. Heninger et al. (2012) factored ~0.2% of live TLS keys
this way. Naive scan is O(k²) GCDs; the real attack uses a product/remainder tree
for near-linear time over *millions* of keys — which is why it was internet-wide.

In [ ]:
print(inspect.getsource(solution.batch_gcd_recover))

### Watch the guarantee fail — live

This reproduces `test_shared_factor_recovers_both_keys` **and**
`test_safe_keys_not_recovered`. **Control Scorecard terms:** "infeasible to factor
`n`" is a real GUARANTEE for each key *alone* — and it fails across a *population*.
The recovered `d`s actually decrypt; the isolated keys stay safe (gcd = 1 with
everyone).

In [ ]:
recovered = solution.batch_gcd_recover(corpus)
truth = corpus["_ground_truth"]["vulnerable_indices"]
assert set(recovered) == set(truth), (sorted(recovered), sorted(truth))
for i, d in recovered.items():
    k = corpus["keys"][i]
    ct = lab.encrypt(1234567890, (k["n"], k["e"]))
    assert lab.decrypt(ct, (k["n"], d)) == 1234567890
safe = [i for i in range(len(corpus["keys"])) if i not in truth]
assert all(i not in recovered for i in safe)
print(f"batch-GCD recovered BOTH shared-factor keys {sorted(recovered)} (each was 'fine' alone)")
print(f"{len(safe)} isolated keys stayed safe")
print()
print("LESSON (scorecard axis 2): 'infeasible to factor n' is a GUARANTEE per key,")
print("CONDITIONAL on p,q from good INDEPENDENT entropy. Weak RNG => shared prime =>")
print("one GCD recovers d. The guarantee holds alone and fails at population scale.")

## Task 3 — timing side channel, and the constant-time fix

Even with perfect keys, `insecure_equal` early-exits at the first mismatched byte,
so its **duration** reveals how long a prefix matched. `timing_attack` recovers the
secret one byte at a time: for each position it times all 256 candidates
(interleaved via `time_guesses`, so CPU drift can't bias one) and keeps the
**slowest** — the correct byte matches one extra position, i.e. one more `AMPLIFY`
loop, before the early exit. `constant_time_equal` examines every byte and never
early-exits, so duration carries no secret. Timing is noisy: the tests use ~41
interleaved rounds so the one-byte signal dominates jitter — do **not** lower it.

In [ ]:
print(inspect.getsource(solution.timing_attack))
print(inspect.getsource(solution.constant_time_equal))
# correctness of the fix (mirror test_constant_time_equal_is_correct)
assert solution.constant_time_equal(b"abcd", b"abcd") is True
assert solution.constant_time_equal(b"abcd", b"abce") is False
assert solution.constant_time_equal(b"abc",  b"abcd") is False
print("constant_time_equal is a correct equality test")

### Watch the guarantee fail — then hold — live

This reproduces `test_timing_attack_recovers_secret` and
`test_constant_time_defeats_timing_attack` on the tests' own 2-byte secret. The
early-exit compare **leaks** and the secret falls to timing alone; the *same*
attack against the constant-time compare recovers nothing. The algorithm didn't
change — the *condition* (constant time) did. Two full 256-candidate scans at 41
rounds take a second or two; that is the price of a stable signal.

In [ ]:
SECRET = bytes([0xA5, 0x3C])

# early-exit compare LEAKS -> attack recovers the secret with no read access
got_leaky = solution.timing_attack(len(SECRET), lab.make_oracle(SECRET, compare=lab.insecure_equal), rounds=41)
assert got_leaky == SECRET, got_leaky.hex()
print(f"early-exit compare leaked: recovered {got_leaky.hex()} by timing alone")

# constant-time compare -> SAME attack recovers nothing
got_ct = solution.timing_attack(len(SECRET), lab.make_oracle(SECRET, compare=solution.constant_time_equal), rounds=41)
assert got_ct != SECRET, "constant-time compare should leak nothing"
print(f"constant-time compare leaked nothing: same attack got {got_ct.hex()} != {SECRET.hex()}")
print()
print("LESSON (scorecard axis 2): a secret compare has NO confidentiality guarantee")
print("unless it runs in CONSTANT TIME. Variable-time handling leaks the secret;")
print("the fix is a condition on the implementation, not a stronger primitive.")